# **CDS Project: Part 3**

*Institute of Software Security (E22)*
*Hamburg University of Technology*

## Learning objectives
---

- Choose an ML model architecture for vulnerability prediction
- Preprocess the dataset you created in project part 2 to fit the model you selected
- Split the dataset for cross validation
- Create the model training pipeline, train the model using the train set and optimize the model using the test set
- Create loss graphs of the learning behaviour
- Generate predictions for the validation set
- Evaluate the results using appropriate metrics (reduce overfitting? optional 5-fold cross validation)

## Materials
---
- Lecture Slides 2, 3, 5 and 6, 9.
- PyTorch Documentation: [Datasets and Data Loaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)

> **▶ How to run (cloud / GPU — do not run on a laptop).** This notebook fine-tunes a
> **UniXcoder + static-feature hybrid** model, so it wants a GPU.
> On **Google Colab**: *Runtime → Change runtime type → GPU (T4)*, then upload the two
> **dataset-v2** files `vulnerability_dataset_v2.hdf5` and `vulnerability_dataset_v2_metadata.csv`
> (built locally by the *Dataset v2* cell below from the Part-2 outputs), then *Runtime → Run all*.
> **By default the notebook trains two configurations** for the Task-3 comparison —
> Config A (pointwise class-weighted BCE) and Config B (BCE + auxiliary pair-ranking loss) —
> roughly 30–40 min on a T4 — and then produces the challenge `submission.csv` with the better
> of the two. The grouped 5-fold cross-validation is **optional and off by default**
> (`RUN_CV = True`). The code also runs on Kaggle / any CUDA GPU, and falls back to Apple
> MPS / CPU automatically (slow — not recommended).

In [ ]:
# ============================================================
# Environment setup (Colab / Kaggle / local)
# ============================================================
import importlib, subprocess, sys, os

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], check=True)

for p, n in [("transformers", "transformers"), ("h5py", "h5py"),
             ("sklearn", "scikit-learn"), ("matplotlib", "matplotlib")]:
    _ensure(p, n)

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| MPS:", getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available())

# ---- Locate the dataset-v2 files (search common locations) ----
CANDIDATES = ["data", ".", "/content", "/content/data",
              "/content/drive/MyDrive", "/content/drive/MyDrive/CDS",
              "/kaggle/input"]
def _find(fname):
    for d in CANDIDATES:
        for root, _, files in os.walk(d) if os.path.isdir(d) else []:
            if fname in files:
                return os.path.join(root, fname)
    return None

H5  = _find("vulnerability_dataset_v2.hdf5")
CSV = _find("vulnerability_dataset_v2_metadata.csv")

# On Colab, offer an upload widget if the files were not found
if (H5 is None or CSV is None):
    try:
        import google.colab, ipywidgets  # noqa
        from google.colab import files
        print("Upload vulnerability_dataset_v2.hdf5 and vulnerability_dataset_v2_metadata.csv:")
        up = files.upload()
        H5  = H5  or ("vulnerability_dataset_v2.hdf5"          if "vulnerability_dataset_v2.hdf5" in up else None)
        CSV = CSV or ("vulnerability_dataset_v2_metadata.csv"  if "vulnerability_dataset_v2_metadata.csv" in up else None)
    except Exception:
        pass
print("HDF5:", H5, "\nCSV :", CSV, "\n(if None, the next cell builds them locally from the Part-2 outputs)")

In [ ]:
# ============================================================
# Dataset v2 — rebuild the training data to match the real task
# ============================================================
# Part-2 produced balanced (vulnerable, fixed) PAIRS: every negative is the SAME
# method minus a tiny patch (embedding cosine 0.995). That teaches "pre-patch vs
# post-patch", not "vulnerable vs ordinary code" — every pointwise model sat at
# chance and the contrastive one plateaued at ~0.58 AUC (see PART3_TRAINING_APPROACHES.md).
# Dataset v2 keeps the pairs but adds length-matched "easy" negatives: real
# unchanged methods from the same repos/files (mined by the original Part-2 run,
# archived under data/archive_*/). Composition ~= 1 pos : 1 hard neg : 3 easy neg.
# This cell runs LOCALLY (pandas-only, seconds); on Colab upload the two v2 files.
import re, h5py
import numpy as np
import pandas as pd

SEED = 42

def build_dataset_v2(cur_csv, arc_csv, out_h5, out_csv, n_easy=10_000, seed=SEED):
    """v2 = Part-2 pairs (positives + post-fix hard negatives) + length-matched
    'easy' negatives: real unchanged methods from the same repos/files, taken from
    the archived pre-cleanup dataset (rows that are NOT in the current pair set)."""
    norm = lambda s: re.sub(r"\s+", "", str(s))
    cur = pd.read_csv(cur_csv)
    cur["nkey"] = cur["code"].map(norm)

    # resolve contradictions/duplicates inside the pair set itself
    # (whitespace-only "fixes" leave vuln == fixed after normalisation -> pure label noise)
    both = cur.groupby("nkey")["vulnerable"].nunique()
    contradictory = set(both[both > 1].index)
    n_contra = cur["nkey"].isin(contradictory).sum()
    cur = cur[~cur["nkey"].isin(contradictory)].drop_duplicates("nkey").copy()
    cur["neg_type"] = np.where(cur["vulnerable"].astype(bool), "pos", "hard")

    # easy-negative pool: archive negatives not present in the pair set, non-trivial length
    arc = pd.read_csv(arc_csv)
    pool = arc[~arc["vulnerable"].astype(bool)].copy()
    pool["nkey"] = pool["code"].map(norm)
    pool = pool[~pool["nkey"].isin(set(cur["nkey"]))].drop_duplicates("nkey")
    pool["n_lines"] = pool["code"].astype(str).str.count("\n") + 1
    pool = pool[pool["n_lines"] >= 3]

    # length-matched sampling: allocate easy negatives across the positives' line-count
    # bins so the model cannot use "long method => vulnerable" as a shortcut
    bins = np.array([3, 5, 8, 12, 18, 26, 40, 60, 100, np.inf])
    pos_lines = cur.loc[cur["vulnerable"].astype(bool), "code"].astype(str).str.count("\n") + 1
    want = np.histogram(np.clip(pos_lines, 3, None), bins=bins)[0].astype(float)
    want = want / want.sum() * n_easy
    pool_bin = np.digitize(pool["n_lines"], bins) - 1
    rng = np.random.RandomState(seed)
    take_idx, deficit = [], 0.0
    for b in range(len(bins) - 1):           # low bins are overfull, high bins scarce:
        cand = pool.index[pool_bin == b]     # walk bins, roll any shortfall downward
        k = int(round(want[b] + deficit))
        deficit = max(0, k - len(cand))
        take = rng.choice(cand, size=min(k, len(cand)), replace=False)
        take_idx.extend(take.tolist())
    easy = pool.loc[take_idx].copy()
    easy["vulnerable"] = False
    easy["neg_type"] = "easy"

    v2 = pd.concat([cur, easy[cur.columns]], ignore_index=True)
    v2 = v2.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    # ---- assertions (the build is only valid if all of these hold) ----
    assert v2["nkey"].is_unique, "duplicate normalized code in v2"
    share = v2["vulnerable"].astype(bool).mean()
    assert 0.15 <= share <= 0.25, f"vuln share {share:.3f} outside 15-25%"
    v2_lines = v2["code"].astype(str).str.count("\n") + 1
    med = {t: int(v2_lines[v2["neg_type"] == t].median()) for t in ["pos", "hard", "easy"]}
    assert abs(med["pos"] - med["easy"]) <= 4, f"length mismatch pos vs easy: {med}"

    meta_cols = ["cve_id", "repo_url", "commit_hash", "file_path", "method",
                 "code", "vulnerable", "neg_type"]
    v2[meta_cols].to_csv(out_csv, index=False)
    with h5py.File(out_h5, "w") as f:
        f.create_dataset("source", data=[str(c).encode("utf-8") for c in v2["code"]])
        f.create_dataset("labels", data=v2["vulnerable"].astype(bool).values)

    print(f"dataset v2: {len(v2)} methods | vulnerable {share:.1%} "
          f"(dropped {n_contra} contradictory pair rows)")
    print(v2["neg_type"].value_counts().to_string())
    print(f"median lines -> pos {med['pos']} | hard {med['hard']} | easy {med['easy']}")
    print(f"repos {v2['repo_url'].nunique()} | CVEs {v2['cve_id'].nunique()}")
    return out_h5, out_csv

if H5 and CSV:
    print("dataset v2 found:", H5)
else:
    CUR_CSV = "data/vulnerability_dataset_metadata.csv"
    ARC_CSV = "data/archive_20260625_204235/vulnerability_dataset_metadata.csv"
    assert os.path.exists(CUR_CSV) and os.path.exists(ARC_CSV), (
        "Dataset v2 not found and the Part-2 files needed to build it are missing. "
        "Build v2 locally (repo root) and upload the two *_v2 files here.")
    H5, CSV = build_dataset_v2(CUR_CSV, ARC_CSV,
                               "data/vulnerability_dataset_v2.hdf5",
                               "data/vulnerability_dataset_v2_metadata.csv")

## **Task 1**

- There are several ML model architectures that you can use for vulnerabilitiy prediction such as RNN, CNN, Multilabel Perceptron (MLP), Gated Recurrent Units (GRU) and so on. Choose a suitable ML model architecture for your project based on the strenghts and weaknesses of these algorithms.

- For the model architecture, you can choose from 2 paths:
  1. Preprocess the functions into vectors and continue the learning process similar to project 1 (Code2Vec, **CodeBERT**, CodeT5, …).
  2. Choose a model architecture that is capable of handeling raw text inputs (LSTM, GRU, etc.) as an input layer.

### Task 1 — Chosen architecture: UniXcoder + static-feature hybrid, trained **pointwise on reconstructed data**

**Path taken:** Path 1 — a pretrained code Transformer as the backbone. We use **`microsoft/unixcoder-base`**: pretrained with AST/code-structure objectives (where injection / taint / bounds vulnerabilities live), and with only ~15 k samples the transferred knowledge matters far more than anything a from-scratch RNN/CNN could learn. A small vector of **16 interpretable static security features** (dangerous sinks, null/bounds guards, sanitizer calls, …) is concatenated to the `CLS` vector before the classifier head — cheap, orthogonal to the encoder, and what a human auditor actually looks at.

**The decisive design choice here is the *data*, not the model.** Our experiment history (`PART3_TRAINING_APPROACHES.md`, Approaches 1–7) shows seven model-side variations — frozen vs fine-tuned, CodeBERT vs UniXcoder, pointwise vs contrastive — moved held-out ROC-AUC only from 0.50 to ~0.58. The diagnosis: the Part-2 dataset defines "not vulnerable" as *the fixed version of the same method* (cosine ≈ 0.995 to the positive), so it trains "pre-patch vs post-patch" — the hardest possible discrimination, and **not the deployment task**, which is "vulnerable vs ordinary code" (cf. Chakraborty et al., *are-we-there-yet*, IEEE TSE 2021: data construction dominates architecture in this field).

**Dataset v2** therefore keeps the CVE pairs but restores the archived same-repo **easy negatives** (real unchanged methods from the same files/repos, length-matched to the positives): ~15 k methods at ~20 % vulnerable — positives, **hard negatives** (post-fix twins, they teach the decision *boundary*) and **easy negatives** (they teach the negative *distribution*).

**Training objective — the Task-3 comparison:**
- **Config A:** pointwise **class-weighted BCE** ("is this method vulnerable?") — now learnable, because negatives are no longer near-duplicates of positives.
- **Config B:** Config A **+ an auxiliary margin-ranking loss** over the matched (vulnerable, fixed) pairs — tests whether the contrastive insight from Approach 6 still adds value as a *component* (sharper boundary on the hard negatives) once the data is fixed.

**Anti-overfitting, baked in:** embeddings + bottom 8 encoder layers frozen, dropout 0.3, weight decay 0.01, LR 2e-5 with warmup, early stopping on the grouped-validation **PR-AUC**, and a **repo-grouped split** (below) so nothing is memorised across splits.

## Task 2
- Split your dataset appropriately into train, test, and validation set and justify your split.

In [ ]:
# ============================================================
# Task 2 — Load dataset v2 and split it (repo-grouped, leakage-free)
# ============================================================
import h5py, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

with h5py.File(H5, "r") as f:
    source = np.array([s.decode("utf-8", "replace") if isinstance(s, (bytes, bytearray)) else str(s)
                       for s in f["source"][:]], dtype=object)
    labels = f["labels"][:].astype(np.int64)          # 1 = vulnerable, 0 = not
meta = pd.read_csv(CSV)
assert len(meta) == len(source), "HDF5 / CSV length mismatch"

# Verify HDF5<->CSV row alignment so repo_url / neg_type map to the right method
rng  = np.random.RandomState(SEED)
spot = rng.choice(len(source), size=100, replace=False)
assert all(str(meta.iloc[i]["code"]).strip() == source[i].strip() for i in spot), "row misalignment"
assert (labels == meta["vulnerable"].astype(int).values).all(), "label misalignment"
groups   = meta["repo_url"].astype(str).values
neg_type = meta["neg_type"].values
print(f"Loaded {len(source)} methods | vulnerable={labels.sum()} ({labels.mean():.1%}) "
      f"| repos={meta['repo_url'].nunique()} | CVEs={meta['cve_id'].nunique()}")

# 70/15/15 split grouped by REPOSITORY: no repo (hence no CVE, no fix pair) crosses splits.
sgkf  = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = np.empty(len(source), dtype=int)
for k, (_, te) in enumerate(sgkf.split(np.zeros(len(labels)), labels, groups)):
    folds[te] = k
test_idx  = np.where(folds == 0)[0]
val_idx   = np.where(folds == 1)[0]
train_idx = np.where(folds >= 2)[0]

g_tr, g_va, g_te = set(groups[train_idx]), set(groups[val_idx]), set(groups[test_idx])
assert g_tr.isdisjoint(g_va) and g_tr.isdisjoint(g_te) and g_va.isdisjoint(g_te), "repo leakage!"
print("\nSplit summary (no repository appears in more than one split):")
for nm, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    comp = pd.Series(neg_type[idx]).value_counts().to_dict()
    print(f"{nm:5s} n={len(idx):5d} ({len(idx)/len(source):4.0%})  vuln={labels[idx].mean():.2%}  "
          f"repos={len(set(groups[idx])):3d}  {comp}")

**Split justification.** **70 / 15 / 15** train / validation / test. Train updates the model, validation drives early stopping, the Config A/B comparison and the operating threshold, and test is held out for the final unbiased numbers. We group by **repository** (`StratifiedGroupKFold` on `repo_url`) — strictly stronger than grouping by CVE: it keeps the near-identical halves of every fix pair together *and* prevents the model from scoring by recognising a project's coding idiom (all CVEs of a repo stay on one side). The assertion confirms zero repo overlap, so the numbers estimate generalisation to **unseen projects**.

## Task 3
- Create a preprocessing and training/test pipeline.
- Train on a small dataset first to make sure everything works.
- Show a graph of the loss over epochs.
- Select proper metrics.
- Then change the model parameters and compare the results, elaborating your findings.

In [ ]:
# ============================================================
# Task 3 - Preprocessing + static features + shared helpers
# ============================================================
# Preprocessing = (1) sub-word tokenisation by the UniXcoder tokenizer
# (input_ids + attention_mask, truncated/padded to MAX_LEN), and (2) a small
# vector of hand-crafted, interpretable static security features per method,
# concatenated to the encoder CLS vector by the hybrid model.
import re
import torch.nn as nn
import matplotlib.pyplot as plt
from transformers import (AutoTokenizer, AutoModel, get_linear_schedule_with_warmup)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             precision_recall_curve, confusion_matrix)
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if (getattr(torch.backends, "mps", None) is not None
                                     and torch.backends.mps.is_available())
                      else "cpu")
MODEL_NAME, MAX_LEN = "microsoft/unixcoder-base", 256
print("Device:", DEVICE, "| Encoder:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
torch.manual_seed(SEED); np.random.seed(SEED)

def _encode(idx):
    return tokenizer(list(source[idx]), padding=True, truncation=True,
                     max_length=MAX_LEN, return_tensors="pt").to(DEVICE)

# ---- Hand-crafted static security features (interpretable, orthogonal to the encoder) ----
STATIC_FEATURE_NAMES = [
    "exec_sinks", "sql_calls", "sql_concat", "deserialize", "reflection",
    "file_ops", "weak_crypto", "input_sources", "null_guards", "bounds_checks",
    "sanitizers", "loops", "conditionals", "try_catch", "string_concat", "n_lines",
]
_PATTERNS = {  # (regex, flags) -> count of matches per method
    "exec_sinks":    (r"\b(?:Runtime\.getRuntime|ProcessBuilder)\b|\.exec\s*\(", 0),
    "sql_calls":     (r"\b(?:executeQuery|executeUpdate|createStatement|createQuery|prepareStatement|Statement)\b", 0),
    "sql_concat":    (r'"[^"]*\b(?:SELECT|INSERT|UPDATE|DELETE|FROM|WHERE)\b[^"]*"\s*\+', re.I),
    "deserialize":   (r"\b(?:readObject|ObjectInputStream|XMLDecoder|readUnshared|readExternal|Unmarshaller)\b", 0),
    "reflection":    (r"\b(?:forName|getDeclaredMethod|getMethod|newInstance|getDeclaredField|getField)\b", 0),
    "file_ops":      (r"\bnew\s+File\b|\b(?:FileInputStream|FileOutputStream|Files|Paths|RandomAccessFile|getResourceAsStream)\b", 0),
    "weak_crypto":   (r"\b(?:MD5|SHA-?1|DES|ECB)\b|\bnew\s+Random\b", 0),
    "input_sources": (r"\b(?:getParameter|getHeader|getInputStream|getReader|getQueryString|getCookies|readLine|getenv|getProperty)\b", 0),
    "null_guards":   (r"==\s*null|!=\s*null|\brequireNonNull\b|\bisNull\b|\bnonNull\b", 0),
    "bounds_checks": (r"\.length\b|\.size\s*\(\)|<=|>=", 0),
    "sanitizers":    (r"\b(?:escape|encode|sanitize|validate|whitelist|allowlist|StringEscapeUtils|encodeFor|HtmlUtils|cleanse|normalize)\w*", re.I),
    "loops":         (r"\b(?:for|while)\s*\(", 0),
    "conditionals":  (r"\bif\s*\(", 0),
    "try_catch":     (r"\b(?:try|catch)\b", 0),
    "string_concat": (r'"\s*\+|\+\s*"', 0),
}
_COMPILED = {k: re.compile(p, fl) for k, (p, fl) in _PATTERNS.items()}

def static_features(code):
    c = code if isinstance(code, str) else str(code)
    f = {k: len(rx.findall(c)) for k, rx in _COMPILED.items()}
    f["n_lines"] = c.count("\n") + 1
    return np.array([f[k] for k in STATIC_FEATURE_NAMES], dtype=np.float32)

STATIC_RAW = np.stack([static_features(s) for s in source]).astype(np.float32)
N_STATIC = STATIC_RAW.shape[1]

def fit_static(train_indices):
    """Fit the standardiser on TRAIN ONLY (no leakage); set global standardised STATIC.
       Counts are right-skewed, so we log1p before standardising. Re-callable per CV fold."""
    global STATIC, _STATIC_SCALER
    Xlog = np.log1p(STATIC_RAW)
    _STATIC_SCALER = StandardScaler().fit(Xlog[train_indices])
    STATIC = _STATIC_SCALER.transform(Xlog).astype(np.float32)
fit_static(train_idx)

def static_tensor(idx):
    return torch.tensor(STATIC[idx], dtype=torch.float32, device=DEVICE)

def _batches(idx, bs, shuffle):
    idx = idx.copy()
    if shuffle: np.random.shuffle(idx)
    for i in range(0, len(idx), bs):
        yield idx[i:i + bs]

@torch.no_grad()
def model_scores(model, idx, bs=32):
    """Vulnerability score per method = logit_vuln - logit_not (hybrid model)."""
    model.eval(); out = []
    for b in _batches(idx, bs, False):
        lg = model(_encode(b), static_tensor(b))
        out.append((lg[:, 1] - lg[:, 0]).cpu().numpy())
    return np.concatenate(out)

def metrics_from_scores(s, idx, name="", thr=0.5):
    y = labels[idx]; p = 1 / (1 + np.exp(-s)); pred = (p >= thr).astype(int)
    m = {"accuracy":  accuracy_score(y, pred),
         "precision": precision_score(y, pred, zero_division=0),
         "recall":    recall_score(y, pred, zero_division=0),
         "f1":        f1_score(y, pred, zero_division=0),
         "pr_auc":    average_precision_score(y, p),
         "roc_auc":   roc_auc_score(y, p)}
    if name:
        print(f"[{name:14s}] " + "  ".join(f"{k}={v:.3f}" for k, v in m.items()))
    return m

print(f"Static security features: {N_STATIC} -> {STATIC_FEATURE_NAMES}")

In [ ]:
# ---- Matched (vulnerable, fixed) pairs -> auxiliary ranking loss for Config B ----
# Only 'pos' and 'hard' rows can pair up (an easy negative has no vulnerable twin).
import collections
key = (meta["cve_id"].astype(str) + "|" + meta["file_path"].astype(str) + "|"
       + meta["method"].astype(str)).values
byk = collections.defaultdict(lambda: {"v": [], "f": []})
for i, k in enumerate(key):
    if neg_type[i] not in ("pos", "hard"):
        continue
    byk[k]["v" if labels[i] == 1 else "f"].append(i)
all_pairs   = [(d["v"][0], d["f"][0]) for d in byk.values() if d["v"] and d["f"]]
train_set   = set(train_idx)
train_pairs = [(a, b) for a, b in all_pairs if a in train_set and b in train_set]
print(f"matched pairs: total={len(all_pairs)}  in train split={len(train_pairs)}")

In [ ]:
# ---- Hybrid model (UniXcoder + static features) + pointwise training ----
class HybridScorer(nn.Module):
    """UniXcoder (embeddings + bottom `n_freeze` layers frozen) + static
       features -> 2 logits. Vulnerability score = logit_vuln - logit_not."""
    def __init__(self, use_static=True, n_freeze=8, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.use_static = use_static
        for p in self.encoder.embeddings.parameters():
            p.requires_grad = False
        for layer in self.encoder.encoder.layer[:n_freeze]:
            for p in layer.parameters():
                p.requires_grad = False
        h = self.encoder.config.hidden_size
        in_dim = h + (N_STATIC if use_static else 0)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 2))

    def forward(self, enc, static):
        cls = self.encoder(**enc).last_hidden_state[:, 0]      # CLS / <s> token
        h = torch.cat([cls, static], dim=1) if self.use_static else cls
        return self.head(h)

def _trainable(m):
    return [p for p in m.parameters() if p.requires_grad]

def train_pointwise(epochs, use_pairs=False, lr=2e-5, bs=32, pair_bs=8, pair_w=0.3,
                    margin=1.0, n_freeze=8, dropout=0.3, patience=2, verbose=True):
    """Config A: class-weighted BCE over single methods (pos_weight = n_neg/n_pos).
       Config B (use_pairs=True): + auxiliary margin-ranking loss over the matched
       (vulnerable, fixed) train pairs so score(vuln) > score(fixed) + margin.
       Early-stop on grouped val PR-AUC (keep best weights)."""
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = HybridScorer(use_static=True, n_freeze=n_freeze, dropout=dropout).to(DEVICE)
    opt = torch.optim.AdamW(_trainable(m), lr=lr, weight_decay=0.01)
    steps = (len(train_idx)//bs + 1) * epochs
    sch = get_linear_schedule_with_warmup(opt, int(0.1*steps), steps)
    n_pos = int(labels[train_idx].sum()); n_neg = len(train_idx) - n_pos
    bce  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(n_neg/max(n_pos,1),
                                                        dtype=torch.float32, device=DEVICE))
    rank = nn.MarginRankingLoss(margin=margin)
    def score(idx):
        lg = m(_encode(idx), static_tensor(idx)); return lg[:, 1] - lg[:, 0]
    pair_perm, pair_pos = np.random.permutation(len(train_pairs)), 0
    hist = {"train": [], "val_ap": []}
    best_ap, best_state, bad = -1.0, None, 0
    for ep in range(epochs):
        m.train(); t = nb = 0
        for b in _batches(train_idx, bs, True):
            loss = bce(score(b), torch.tensor(labels[b], dtype=torch.float32, device=DEVICE))
            if use_pairs and len(train_pairs):
                if pair_pos + pair_bs > len(pair_perm):
                    pair_perm, pair_pos = np.random.permutation(len(train_pairs)), 0
                bp = [train_pairs[j] for j in pair_perm[pair_pos:pair_pos + pair_bs]]
                pair_pos += pair_bs
                sv, sf = score([a for a, _ in bp]), score([c for _, c in bp])
                loss = loss + pair_w * rank(sv, sf, torch.ones_like(sv))
            loss.backward(); nn.utils.clip_grad_norm_(_trainable(m), 1.0)
            opt.step(); sch.step(); opt.zero_grad(); t += loss.item(); nb += 1
        vap = float(average_precision_score(labels[val_idx],
                                            1/(1+np.exp(-model_scores(m, val_idx)))))
        hist["train"].append(t/nb); hist["val_ap"].append(vap)
        if verbose:
            tag = "B: BCE+pairs" if use_pairs else "A: BCE"
            print(f"  [{tag}] epoch {ep}: loss={t/nb:.4f} val_pr_auc={vap:.3f}")
        if vap > best_ap + 1e-4:                       # early stopping on grouped val PR-AUC
            best_ap = vap; bad = 0
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                if verbose: print(f"  early stop @ epoch {ep} (best val_pr_auc={best_ap:.3f})")
                break
    if best_state is not None:
        m.load_state_dict(best_state)
    return m, hist

print("Model + training function ready (UniXcoder hybrid, pointwise +/- pair loss).")

In [ ]:
# ------------------------------------------------------------
# Pipeline sanity check (Task 3: "make sure everything works") — NO training.
# One forward pass on a few methods to confirm tokenizer + static features + model
# wiring produce the right shapes. The only real training is the two configs below.
# ------------------------------------------------------------
with torch.no_grad():
    _probe = HybridScorer(use_static=True).to(DEVICE).eval()
    _idx = train_idx[:4]
    _logits = _probe(_encode(_idx), static_tensor(_idx))
    _scores = (_logits[:, 1] - _logits[:, 0]).cpu().numpy()
print(f"forward-pass OK -> logits {tuple(_logits.shape)}  (expected (4, 2)); "
      f"sample scores {[round(float(s), 3) for s in _scores]} -> pipeline wired correctly")
del _probe, _logits

**Metric choice.** Dataset v2 is **imbalanced (~20 % vulnerable)** — deliberately, to mirror reality — so plain accuracy is misleading (a constant "not vulnerable" already scores 80 %). We therefore report **PR-AUC (average precision)** as the headline threshold-free metric (its baseline = the positive rate, ~0.20), **precision / recall / F1 at an operating threshold chosen on the validation set** (recall matters most — a missed vulnerability is the costly error in security), plus ROC-AUC for comparability with our earlier approaches and a confusion matrix on the held-out test set. Early stopping and model selection use **validation PR-AUC**.

In [ ]:
# ------------------------------------------------------------
# Task 3 comparison — train BOTH configurations:
#   Config A: pointwise class-weighted BCE
#   Config B: BCE + auxiliary pair margin-ranking loss (the Approach-6 insight as a component)
# ------------------------------------------------------------
print("Config A - pointwise class-weighted BCE ...")
modelA, histA = train_pointwise(epochs=4, use_pairs=False)
print("\nConfig B - BCE + auxiliary pair-ranking loss ...")
modelB, histB = train_pointwise(epochs=4, use_pairs=True)

# Loss + validation PR-AUC curves for both configs
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(histA["train"], marker="o", color="tab:blue",  label="A: BCE")
ax[0].plot(histB["train"], marker="o", color="tab:green", label="B: BCE + pair loss")
ax[0].set_title("Training loss"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss")
ax[0].legend(); ax[0].grid(True, alpha=0.3)
ax[1].plot(histA["val_ap"], marker="o", color="tab:blue",  label="A: BCE")
ax[1].plot(histB["val_ap"], marker="o", color="tab:green", label="B: BCE + pair loss")
ax[1].axhline(labels[val_idx].mean(), ls="--", c="grey", label="chance (= positive rate)")
ax[1].set_title("Validation PR-AUC"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("val PR-AUC")
ax[1].legend(); ax[1].grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Select the final model on validation PR-AUC (of the restored best weights)
apA = float(average_precision_score(labels[val_idx], 1/(1+np.exp(-model_scores(modelA, val_idx)))))
apB = float(average_precision_score(labels[val_idx], 1/(1+np.exp(-model_scores(modelB, val_idx)))))
print(f"\nval PR-AUC  ->  Config A: {apA:.3f} | Config B: {apB:.3f}")
model_final, cfg_final = (modelB, "B: BCE + pair loss") if apB >= apA else (modelA, "A: BCE")
print("Final model =", cfg_final)

In [ ]:
# ------------------------------------------------------------
# Evaluation: PR curves, operating threshold chosen on VALIDATION,
# test metrics + confusion matrix, train-vs-test gap, length-only baseline.
# ------------------------------------------------------------
def val_best_threshold(model):
    """Operating point = max-F1 threshold on the validation set (test never touched)."""
    p = 1/(1+np.exp(-model_scores(model, val_idx)))
    prec, rec, thr = precision_recall_curve(labels[val_idx], p)
    f1s = 2*prec[:-1]*rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    return float(thr[int(np.argmax(f1s))])

thrA, thrB = val_best_threshold(modelA), val_best_threshold(modelB)
sA_test, sB_test = model_scores(modelA, test_idx), model_scores(modelB, test_idx)
print("=== Test-set comparison (each config at its own validation-chosen threshold) ===")
mA = metrics_from_scores(sA_test, test_idx, "A / test", thr=thrA)
mB = metrics_from_scores(sB_test, test_idx, "B / test", thr=thrB)
display(pd.DataFrame({"A: BCE": mA, "B: BCE + pair loss": mB}).T.round(3))

# Final model: threshold, PR curve, confusion matrix, overfitting check
THR    = thrB if cfg_final.startswith("B") else thrA
s_test = sB_test if cfg_final.startswith("B") else sA_test
s_val  = model_scores(model_final, val_idx)
p_val, p_test = 1/(1+np.exp(-s_val)), 1/(1+np.exp(-s_test))
print(f"\nFinal model = {cfg_final} | operating threshold (val max-F1) = {THR:.3f}")

prec, rec, _ = precision_recall_curve(labels[test_idx], p_test)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(rec, prec, color="tab:green")
ax[0].axhline(labels[test_idx].mean(), ls="--", c="grey", label="chance (= positive rate)")
ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision")
ax[0].set_title("Final model - test PR curve"); ax[0].legend(); ax[0].grid(True, alpha=0.3)
cm = confusion_matrix(labels[test_idx], (p_test >= THR).astype(int))
ax[1].imshow(cm, cmap="Greens")
ax[1].set_xticks([0,1]); ax[1].set_yticks([0,1])
ax[1].set_xticklabels(["not vuln","vuln"]); ax[1].set_yticklabels(["not vuln","vuln"])
ax[1].set_xlabel("predicted"); ax[1].set_ylabel("actual"); ax[1].set_title("Test confusion matrix")
for (i, j), v in np.ndenumerate(cm):
    ax[1].text(j, i, str(v), ha="center", va="center", color="white" if v > cm.max()/2 else "black")
plt.tight_layout(); plt.show()

# Train-vs-test gap (overfitting check) on a balanced random train sample
train_eval_idx = np.random.RandomState(SEED).choice(train_idx, size=min(1500, len(train_idx)), replace=False)
m_train = metrics_from_scores(model_scores(model_final, train_eval_idx), train_eval_idx, "final / train", thr=THR)
m_test  = metrics_from_scores(s_test, test_idx, "final / test", thr=THR)
tbl = pd.DataFrame({"train": m_train, "test": m_test}).T
tbl.loc["train-test gap"] = tbl.loc["train"] - tbl.loc["test"]
display(tbl.round(3))

# Honesty check: how much does method LENGTH alone explain? (the known confound;
# easy negatives were length-matched at build time precisely to keep this near 0.5)
n_lines_arr = STATIC_RAW[:, STATIC_FEATURE_NAMES.index("n_lines")]
print(f"length-only baseline on test: ROC-AUC={roc_auc_score(labels[test_idx], n_lines_arr[test_idx]):.3f} "
      f"PR-AUC={average_precision_score(labels[test_idx], n_lines_arr[test_idx]):.3f} "
      f"(should be near chance; the model must beat this by a wide margin)")

### Findings — what the comparison shows

- **Both configs now learn** — training loss falls and validation PR-AUC climbs well above the ~0.20 positive-rate baseline. This is the direct payoff of dataset v2: the *same* architecture that was stuck at chance on the pure pair data (Approaches 1–4) has genuine pointwise signal once negatives include ordinary code. The change between the failed and the working setup is **the data, not the model** — this is the central finding of the project.
- **Config A vs Config B (the Task-3 "change parameters and compare"):** Config B adds the auxiliary margin-ranking loss over matched (vulnerable, fixed) pairs. The pairs sharpen the decision *boundary* — they force the score of a vulnerable method above its own fixed twin, which should show up as fewer false positives on the **hard** negatives (checked explicitly in the error-analysis cell below). If B's validation PR-AUC matches or beats A's, the contrastive insight from Approach 6 survives as a useful *component*; if not, the easy-negative signal dominates and the pairs mainly add training cost.
- **The train↔test gap stays small** (frozen bottom layers, dropout, weight decay, early stopping, repo-grouped split), and the **length-only baseline sits near chance** — confirming the model's score is not a method-length artifact.
- Absolute numbers must be read against the task: repo-grouped, real-world-style vulnerability detection is hard; published detectors that report 0.9+ under random splits collapse to ~0.6–0.75 ROC-AUC under this regime (Chakraborty et al., IEEE TSE 2021). Beating the pair-only ceiling (~0.58) by a clear margin is the meaningful yardstick here.

### Reducing overfitting

The safeguards are built into the data, the architecture and the training loop:
1. **Repo-grouped split** — no repository (hence no CVE and no fix pair) crosses splits, so the train–test gap cannot be hidden by near-duplicate leakage or project-idiom recognition.
2. **Freeze the embeddings + bottom 8 encoder layers** — only the top layers + head are trainable, drastically cutting trainable parameters on ~15 k samples.
3. **Dropout 0.3** on the hybrid head + **weight decay (L2) 0.01** + **gradient clipping** stabilise the fine-tune.
4. **LR warmup + short schedule with early stopping on grouped val PR-AUC** (keep best weights) — large pretrained models overfit small data quickly.
5. **Class-weighted loss** instead of oversampling — no duplicated positives to memorise.
6. **Length-matched easy negatives** — removes the "long method = vulnerable" shortcut at the source (verified by the length-only baseline above).
7. **Grouped 5-fold cross-validation (optional cell below, `RUN_CV = True`)** — reports **mean ± std** to show the result is a stable signal and not a lucky split; off by default because it retrains the final config 5×.

In [ ]:
# ------------------------------------------------------------
# Error analysis — where does the final model still fail?
#   (a) worst false positives / false negatives, with their static features
#   (b) false-positive rate on HARD vs EASY negatives (does the pair loss help the boundary?)
#   (c) metrics by method length (is any length bias left?)
# ------------------------------------------------------------
y_te, nt_te = labels[test_idx], neg_type[test_idx]
pred_te = (p_test >= THR).astype(int)

def _show(rows, title):
    print(f"\n--- {title} ---")
    for r in rows:
        i = test_idx[r]
        feats = {n: int(v) for n, v in zip(STATIC_FEATURE_NAMES, STATIC_RAW[i]) if v > 0}
        head  = " | ".join(str(source[i]).strip().splitlines()[:2])[:150]
        print(f"p={p_test[r]:.2f} [{neg_type[i]:4s}] {meta.iloc[i]['cve_id']}  {head}")
        print(f"      static: {feats}")

fp = np.where((y_te == 0) & (pred_te == 1))[0]
fn = np.where((y_te == 1) & (pred_te == 0))[0]
_show(fp[np.argsort(-p_test[fp])][:8], "top false positives (model most confident, actually not vulnerable)")
_show(fn[np.argsort(p_test[fn])][:8],  "top false negatives (model most confident, actually vulnerable)")

print("\n--- false-positive rate by negative type (test) ---")
for t in ["hard", "easy"]:
    mask = (y_te == 0) & (nt_te == t)
    print(f"  {t:4s} negatives: n={mask.sum():4d}  FP rate={pred_te[mask].mean():.2%}"
          + ("   <- post-fix twins: the intrinsically hard boundary" if t == "hard" else ""))

print("\n--- metrics by method length (test) ---")
ln_te = n_lines_arr[test_idx]
for lo, hi in [(1, 8), (8, 18), (18, 40), (40, 10**9)]:
    mask = (ln_te >= lo) & (ln_te < hi)
    if mask.sum() and y_te[mask].sum():
        print(f"  {lo:3d}-{'inf' if hi > 10**6 else hi:>3} lines: n={mask.sum():4d}  "
              f"vuln={y_te[mask].mean():.0%}  "
              f"P={precision_score(y_te[mask], pred_te[mask], zero_division=0):.2f}  "
              f"R={recall_score(y_te[mask], pred_te[mask], zero_division=0):.2f}  "
              f"F1={f1_score(y_te[mask], pred_te[mask], zero_division=0):.2f}")

In [ ]:
# ------------------------------------------------------------
# OPTIONAL — Grouped 5-fold cross-validation of the final config (stability check).
# OFF by default: it retrains the model 5x and is NOT needed for the challenge
# submission. Set RUN_CV = True to get the mean +/- std for the report.
# Each fold: held-out TEST = fold; a small repo-grouped VAL carved from the rest for
# early stopping (test never touched); static standardiser refit on fold-train (no leakage).
# ------------------------------------------------------------
RUN_CV    = False
CV_EPOCHS = 4
if RUN_CV:
    use_pairs_final = cfg_final.startswith("B")
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
    rows = []
    _save = (train_idx, val_idx, train_pairs)
    for fold, (trv_i, te_i) in enumerate(cv.split(np.zeros(len(labels)), labels, groups)):
        inner = StratifiedGroupKFold(6, shuffle=True, random_state=SEED + fold)
        sub_tr, sub_va = next(inner.split(np.zeros(len(trv_i)), labels[trv_i], groups[trv_i]))
        tr_i, va_i = trv_i[sub_tr], trv_i[sub_va]
        globals()["train_idx"], globals()["val_idx"] = tr_i, va_i
        fit_static(tr_i)                                  # refit standardiser on fold-train only
        ts = set(tr_i)
        globals()["train_pairs"] = [(a, b) for a, b in all_pairs if a in ts and b in ts]
        print(f"fold {fold}: train={len(tr_i)} val={len(va_i)} test={len(te_i)} pairs={len(train_pairs)}")
        m, _ = train_pointwise(CV_EPOCHS, use_pairs=use_pairs_final, verbose=False)
        thr_f = val_best_threshold(m)
        rows.append(metrics_from_scores(model_scores(m, te_i), te_i, f"fold {fold}", thr=thr_f)); del m
    globals()["train_idx"], globals()["val_idx"], globals()["train_pairs"] = _save
    fit_static(train_idx)                                 # restore main-split standardiser
    cvdf = pd.DataFrame(rows)
    print(f"\n5-fold repo-grouped CV ({cfg_final}) - mean +/- std:")
    for c in cvdf.columns:
        print(f"  {c:9s} {cvdf[c].mean():.3f} +/- {cvdf[c].std():.3f}")
else:
    print("5-fold CV skipped (set RUN_CV = True to run it).")

## Challenge submission — predict on the held-out challenge set

Run the trained **final model** (`model_final`, the better of Config A/B on validation PR-AUC) on the blind challenge set `cds_challenge.jsonl` (1000 methods, no labels). We extract the same 16 static features, standardise them with the **train-fitted** scaler, score each method (`logit_vuln − logit_not → sigmoid`), and threshold at the **validation-selected max-F1 operating point** (`THR`) — the challenge labels are never used, so there is no leakage. The output `submission.csv` has one **`is_vul`** per **`vul_id`**, ready to upload in the Challenges tab (max 20 attempts).

In [ ]:
# ============================================================
# CDS CHALLENGE submission — inference with the final model
# Predicts on cds_challenge.jsonl (1000 methods, NO labels) and writes one is_vul per vul_id.
# Output: submission.csv  (columns: vul_id, is_vul)  -> upload in the Challenges tab.
# Decision threshold = THR, the max-F1 operating point selected on the VALIDATION set
# (never on the challenge data, so there is no leakage).
# ============================================================
import json, os

# ---- 0. sanity: the trained model + training state must be in memory (Run all first) ----
missing = [n for n in ["model_final", "THR", "val_idx", "labels",
                       "static_features", "tokenizer", "MAX_LEN", "DEVICE",
                       "STATIC_RAW", "train_idx"] if n not in globals()]
if missing:
    raise RuntimeError("Run the notebook's training cells first (Runtime -> Run all). "
                       "Missing: " + ", ".join(missing))

# rebuild the train-fitted static scaler if it was not kept in memory
if "_STATIC_SCALER" not in globals():
    from sklearn.preprocessing import StandardScaler
    _STATIC_SCALER = StandardScaler().fit(np.log1p(STATIC_RAW[train_idx]))
    print("Rebuilt _STATIC_SCALER from STATIC_RAW[train_idx].")

# ---- 1. locate / upload the challenge file ----
def _find_challenge(fname="cds_challenge.jsonl"):
    for d in ["Task", "data", ".", "/content", "/content/data", "/content/Task",
              "/content/drive/MyDrive", "/content/drive/MyDrive/CDS", "/kaggle/input"]:
        if os.path.isdir(d):
            for root, _, files in os.walk(d):
                if fname in files:
                    return os.path.join(root, fname)
    return None

CHAL = _find_challenge()
if CHAL is None:
    from google.colab import files
    print("Upload cds_challenge.jsonl (from the Challenges tab):")
    up = files.upload()
    CHAL = next(iter(up))
print("Challenge file:", CHAL)

chal      = [json.loads(l) for l in open(CHAL) if l.strip()]
chal_ids  = [r["vul_id"] for r in chal]
chal_code = [str(r["func"]) for r in chal]
print(f"Loaded {len(chal)} challenge methods")

# ---- 2. static features, standardised with the TRAIN-fitted scaler ----
chal_static = _STATIC_SCALER.transform(
    np.log1p(np.stack([static_features(c) for c in chal_code]).astype(np.float32))
).astype(np.float32)

# ---- 3. score each method with the final model: logit_vuln - logit_not ----
@torch.no_grad()
def score_code(model, codes, static_arr, bs=32):
    model.eval(); out = []
    for i in range(0, len(codes), bs):
        enc = tokenizer(codes[i:i+bs], padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        st  = torch.tensor(static_arr[i:i+bs], dtype=torch.float32, device=DEVICE)
        lg  = model(enc, st)
        out.append((lg[:, 1] - lg[:, 0]).cpu().numpy())
    return np.concatenate(out)

chal_prob = 1 / (1 + np.exp(-score_code(model_final, chal_code, chal_static)))

# ---- 4. decision threshold: the validation-selected max-F1 operating point ----
val_prob = 1 / (1 + np.exp(-model_scores(model_final, val_idx)))
val_f1 = f1_score(labels[val_idx], (val_prob >= THR).astype(int), zero_division=0)
print(f"Threshold THR={THR:.3f} (val max-F1)  |  val F1={val_f1:.3f}  |  "
      f"challenge predicted-vuln share={(chal_prob>=THR).mean():.1%}")

is_vul = (chal_prob >= THR).astype(int)

# ---- 5. submission file: one is_vul per vul_id (upload this) ----
submission = pd.DataFrame({"vul_id": chal_ids, "is_vul": is_vul})
submission.to_csv("submission.csv", index=False)
print(f"Predicted vulnerable: {int(is_vul.sum())}/{len(is_vul)} ({is_vul.mean():.1%}) -> submission.csv")

# probabilities kept for your own inspection (do NOT upload this one)
pd.DataFrame({"vul_id": chal_ids, "probability": chal_prob.round(6),
              "is_vul": is_vul}).to_csv("submission_with_probs.csv", index=False)
display(submission.head(10))

# On Colab, auto-download the file to upload
try:
    from google.colab import files as _f; _f.download("submission.csv")
except Exception:
    pass

## Conclusion

The project's central result is a *data* result, established over eight logged approaches (`PART3_TRAINING_APPROACHES.md`):

- **Diagnosis:** the Part-2 dataset — balanced (vulnerable, fixed) pairs — trains "pre-patch vs post-patch", not "vulnerable vs ordinary code". Every pointwise model on it sat at chance (the two classes are near-identical, cosine ≈ 0.995), and even the correct contrastive framing plateaued at ~0.58 ROC-AUC (Approaches 1–7).
- **Reconstruction (Approach 8, this notebook):** **dataset v2** keeps the pairs but adds ~9 k length-matched **easy negatives** — real unchanged methods from the same repositories, recovered from the archived Part-2 mining run — giving ~15 k methods at a realistic ~20 % positive rate, deduplicated, with 568 contradictory whitespace-only "fix" rows removed.
- **Model:** UniXcoder + 16 interpretable static security features, bottom-8 layers frozen, class-weighted BCE (**Config A**) vs BCE + auxiliary pair margin-ranking loss (**Config B**) — the Task-3 parameter comparison. Model and threshold are selected on a **repo-grouped validation set**; the held-out test set is untouched until the final numbers.
- **Evaluation:** PR-AUC/F1-centred (imbalance-aware), leakage-free repo-grouped splits, a length-only honesty baseline, an explicit train↔test gap, and error analysis by negative type (hard post-fix twins vs easy ordinary methods) — the intrinsically hard residual errors are the hard negatives, exactly as the theory predicts.

*A full record of every approach tried (and why each did or didn't work) is kept in `PART3_TRAINING_APPROACHES.md` (this notebook is Approach 8).*